In [30]:

import numpy as np

# Qiskit
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_histogram, plot_state_city
import qiskit.quantum_info as qi
## real noise data
from qiskit_ibm_runtime.fake_provider import FakeBrisbane as fake
device_backend = fake()


In [31]:
SHOTS = 10000 # How many times to run the circuit.
NUM_QUBITS = 9 # How many physical qubits to use. Must be a multiple of PHYSICAL_PER_LOGICAL.
PHYSICAL_PER_LOGICAL = 3 # How many physical qubits per logical qubit. Must be an odd number.
FILENAME = 'bitstream' # Filename to save the bitstream to.

assert NUM_QUBITS % PHYSICAL_PER_LOGICAL == 0
assert PHYSICAL_PER_LOGICAL % 2 == 1
assert (SHOTS * (NUM_QUBITS / PHYSICAL_PER_LOGICAL)) % 8 == 0 # Must be a multiple of 8 for the bitstream to be byte-aligned

In [32]:
circ = QuantumCircuit(NUM_QUBITS)
circ.h(range(NUM_QUBITS))
circ.measure_all()

In [33]:
sim_ideal = AerSimulator()
sim_fake = AerSimulator.from_backend(device_backend)

# Transpile for noisy gates
circ = transpile(circ, sim_fake)

# Run and get counts
# shots = #tries? , # memory makes it return list of measurements
result = sim_fake.run(circ).result()
print(result[0]
counts = result.get_counts(0)


SyntaxError: '(' was never closed (3646629509.py, line 10)

In [35]:
# Get backend and transpile circuit
TYPE = 'simulated_noise'
if TYPE == 'simulated_noise':
    device_backend = fake()
    sim_fake = AerSimulator.from_backend(device_backend)
    
    # Transpile for noisy gates
    circ = transpile(circ, sim_fake)
    
    # Run the circuit
    result = sim_fake.run(circ, shots=SHOTS).result()
    counts = result.get_counts(0)
    print(counts)
    
    # Convert counts to bitstrings
    bitstrings = list(counts.keys())
    
    # Perform a logical qubit majority vote
    bitstream = []
    for bitstring in bitstrings:
        bitstring = np.array([int(c) for c in bitstring]).reshape(-1, PHYSICAL_PER_LOGICAL)
        majority_votes = (np.mean(bitstring, axis=1) > 0.5).astype(int).tolist()
        bitstream.extend(majority_votes)
    
    bitstring = [str(bit) for bit in bitstream]
    output = [bitstring[i:i+8] for i in range(0, len(bitstring), 8)]
    ba = bytearray(int(''.join(byte), 2) for byte in output)
    bs = bytes(ba)
    
    with open(f"./{FILENAME}.bin", 'wb') as f:
        f.write(bs)
else:
    service = QiskitRuntimeService(channel=CHANNEL, token=os.getenv('IBMQ_API_TOKEN'))
    if CHANNEL == 'ibm_quantum':
        backend = service.least_busy(simulator=False, operational=True)
    else:
        backend = service.least_busy()
    
    pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
    isa_circ = pm.run(circ)
    
    # Initialise Sampler
    sampler = SamplerV2(mode=backend)
    sampler.options.default_shots = SHOTS
    
    # Run the circuit
    job = sampler.run([isa_circ])
    print(f">>> Job ID: {job.job_id()}")
    result = job.result()
    bitstrings = np.array(result[0].data.meas.get_bitstrings())
    
    # Perform a logical qubit majority vote
    bitstream = []
    for bitstring in bitstrings:
        bitstring = np.array([int(c) for c in bitstring]).reshape(-1, PHYSICAL_PER_LOGICAL)
        majority_votes = (np.mean(bitstring, axis=1) > 0.5).astype(int).tolist()
        bitstream.extend(majority_votes)
    
    bitstring = [str(bit) for bit in bitstream]
    output = [bitstring[i:i+8] for i in range(0, len(bitstring), 8)]
    ba = bytearray(int(''.join(byte), 2) for byte in output)
    bs = bytes(ba)
    
    with open(f"./{FILENAME}.bin", 'wb') as f:
        f.write(bs)


{'000111011': 13, '100011010': 21, '110100100': 13, '000000100': 13, '100000111': 25, '111111001': 15, '110001000': 20, '101010001': 15, '100000011': 17, '110011110': 20, '100010011': 16, '100011011': 17, '101001000': 25, '111000101': 14, '010001011': 18, '010110001': 20, '010010101': 16, '101001011': 15, '101100000': 16, '000110010': 24, '101010101': 12, '010000011': 17, '011100001': 14, '010101001': 18, '100010010': 21, '111100011': 18, '111100100': 13, '111001111': 11, '011001110': 18, '100100011': 17, '001000010': 16, '010000100': 20, '101110101': 14, '011110011': 15, '010100100': 18, '011000101': 28, '110010001': 17, '110100111': 20, '010000010': 20, '110110011': 29, '010101011': 16, '101001110': 16, '111001110': 24, '001111010': 22, '001110001': 18, '011011011': 17, '001001101': 18, '001010000': 21, '011000001': 14, '000010001': 24, '001010100': 14, '001110010': 27, '110010111': 16, '001100010': 12, '110011100': 17, '101011000': 15, '110111010': 20, '000111111': 19, '100001101': 